In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer Segment,Market,...,Order Item Total,Order Profit Per Order,Product Price,Shipping Mode,Delay_Gap,Shipping_Pressure_Index,Order_Complexity_Score,Profit_Sensitivity,Discount_Impact,Risk_Category
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,Sporting Goods,Consumer,Pacific Asia,...,314.640015,91.250000,327.75,Standard Class,-1,0.2,327.75,0.277567,13.110000,Low Risk
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,Sporting Goods,Consumer,Pacific Asia,...,311.359985,-249.089996,327.75,Standard Class,1,0.2,327.75,-0.757688,16.387500,High Risk
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,Sporting Goods,Consumer,Pacific Asia,...,309.720001,-247.779999,327.75,Standard Class,0,0.2,327.75,-0.753703,19.665000,Low Risk
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,Sporting Goods,Home Office,Pacific Asia,...,304.809998,22.860001,327.75,Standard Class,-1,0.2,327.75,0.069536,22.942500,Low Risk
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,Sporting Goods,Corporate,Pacific Asia,...,298.250000,134.210007,327.75,Standard Class,-2,0.2,327.75,0.408243,29.497501,Low Risk


In [3]:
X = df.drop(["Late_delivery_risk", "Risk_Category"], axis=1)
y = df["Late_delivery_risk"]

In [4]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", numeric_cols)

Categorical: ['Type', 'Delivery Status', 'Category Name', 'Customer Segment', 'Market', 'Order Region', 'Order State', 'Order Status', 'Shipping Mode']
Numerical: ['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Price', 'Delay_Gap', 'Shipping_Pressure_Index', 'Order_Complexity_Score', 'Profit_Sensitivity', 'Discount_Impact']


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Logistic Regression
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     16303
           1       1.00      1.00      1.00     19791

    accuracy                           1.00     36094
   macro avg       1.00      1.00      1.00     36094
weighted avg       1.00      1.00      1.00     36094



In [8]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Random Forest
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     16303
           1       1.00      1.00      1.00     19791

    accuracy                           1.00     36094
   macro avg       1.00      1.00      1.00     36094
weighted avg       1.00      1.00      1.00     36094



In [9]:
cm = confusion_matrix(y_test, y_pred)
cm

array([[16303,     0],
       [    0, 19791]])

In [10]:
joblib.dump(rf_model, "../models/delivery_risk_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [11]:
def get_risk_category(prob):
    if prob < 0.30:
        return "Low Risk"
    elif prob < 0.70:
        return "Medium Risk"
    else:
        return "High Risk"

In [12]:
sample = X_test.iloc[[0]]

prob = rf_model.predict_proba(sample)[0][1]
prediction = rf_model.predict(sample)[0]

print("Late Delivery Probability:", round(prob * 100, 2), "%")
print("Prediction:", prediction)
print("Risk Category:", get_risk_category(prob))

Late Delivery Probability: 0.0 %
Prediction: 0
Risk Category: Low Risk
